# 01 Corpus and validation

**Question.** What is the AndroCT 2017 eligible population, and which corpus/validation numbers in MASTER_RESULTS.csv are stored in artifacts?

In [ ]:
from pathlib import Path
import os, json, csv
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "chapter_a").is_dir():
    for cand in (Path(".."), Path("../.."), Path("../../..")):
        if (cand.resolve() / "chapter_a").is_dir():
            ROOT = cand.resolve()
            break
os.chdir(ROOT)
MASTER = pd.read_csv(ROOT / "chapter_a" / "MASTER_RESULTS.csv")
ANDROCT = ROOT / "abrg" / "output" / "androct_2017"

def row_eq(mask, artifact_auc):
    sub = MASTER.loc[mask]
    assert len(sub) >= 1, mask
    mval = float(sub.iloc[0]["auc_floor"])
    aval = float(artifact_auc)
    assert round(mval, 6) == round(aval, 6), (mval, aval)


Corpus inventory from post-_CALL_RE `inventory_summary.json` (population), with eligible/split from run2 cache.

In [ ]:
inv = json.loads((ROOT / "datasets/androct_2017/inventory/inventory_summary.json").read_text())
meta = json.loads((ANDROCT / "run2" / "corpus_cache" / "meta.json").read_text())
t1 = pd.read_csv(ROOT / "chapter_a" / "tables" / "T1_corpus.csv")
pop = t1[t1.stage == "population"]
print(pop.to_string(index=False))
for lab in ("benign", "malware"):
    c = inv["classes"][lab]
    row = pop[pop["class"] == lab].iloc[0]
    assert int(row.n_effective) == int(c["n_effective"])
    assert round(float(row.mapped_rate), 6) == round(float(c["mapped_event_rate"]), 6)
    assert int(row.categories_firing) == int(c["n_universe_cats_active"])
print("eligible", meta["n_eligible"], meta["eligibility"]["eligible"])
print("split", {k: len(v) for k, v in meta["split"].items()})


Mapped-event size floor from `run3/floors.json`.

In [ ]:
fl = json.loads((ANDROCT / "run3" / "floors.json").read_text())
floor = fl["mapped_event_count"]["auc_floor"]
row_eq(
    (MASTER.experiment == "run3") & (MASTER.detector == "mapped_event_count"),
    floor,
)
print("mapped floor", floor)


Final cell: MASTER vs artifacts to 6 decimal places.

In [ ]:
assert round(float(MASTER.loc[(MASTER.detector=="mapped_event_count"), "auc_floor"].iloc[0]), 6) == round(floor, 6)
print("ok")
